In [ ]:
# Import libraries
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

pd.set_option('display.max_columns', None)

In [ ]:
# set data dir
DATA_DIR = Path('../data')  # download the files from Zindi into ./data
os.listdir(DATA_DIR)

In [ ]:
# Load files
train = pd.read_csv(DATA_DIR / 'Train.csv')
test = pd.read_csv(DATA_DIR / 'Test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')

# Preview train file
train.head()

In [ ]:
# Preview test file
test.head()

In [ ]:
# Preview sample submission
sample_submission.head()

In [ ]:
# Check shapes of files
train.shape, test.shape, sample_submission.shape

In [ ]:
# Check for duplicates and missing values - train
train.duplicated().any(), train.isnull().sum().any() # Handle missing values by filling them with meadian or any other techniques

In [ ]:
# Check for duplicates and missing values - test
test.duplicated().any(), test.isnull().sum().any() # Handle missing values by filling them with meadian or any other techniques

In [ ]:
# Check unique ids per dataset
train.ID.nunique(), test.ID.nunique()

In [ ]:
# A single ID represents a unique pixel over different time periods
# create a targets_df

train_targets_df = train.groupby('ID')['Target'].first().reset_index()
train_targets_df.head()

In [ ]:
# Check data distribution
train_targets_df.Target.value_counts(normalize = True)

In [ ]:
train.head()

In [ ]:
# Aggrgate pixels over the different time periods to get a wholesome view
# Get columns to aggregate
agg_cols = train.columns.tolist()[2:-1]
agg_cols

In [ ]:
# Define comprehensive aggregation functions
agg_functions = {
    col: [
        'mean',            # Arithmetic mean
        'std',             # Standard deviation
        'min',             # Minimum value
        'max',             # Maximum value
        'median',          # median
    ] for col in agg_cols
}

# Perform the aggregation
aggregated_train = train.groupby('ID').agg(agg_functions)
aggregated_test = test.groupby('ID').agg(agg_functions)

# Rename columns
aggregated_train.columns = [f"{col[0]}_{col[1]}" for col in aggregated_train.columns]
aggregated_test.columns = [f"{col[0]}_{col[1]}" for col in aggregated_test.columns]

# Reset index
aggregated_train = aggregated_train.reset_index()
aggregated_test = aggregated_test.reset_index()

# Preview aggregated train
aggregated_train.head()

In [ ]:
# Preview aggregated test
aggregated_test.head()

In [ ]:
# Number of new generated features
aggregated_train.shape, aggregated_test.shape

In [ ]:
# Add back the train targets to the aggregated df
aggregated_train = aggregated_train.merge(train_targets_df, how = 'left', on = 'ID')

# Preview
aggregated_train.head()

In [ ]:
# Split data for training and validation
X_train, X_val, y_train, y_val= train_test_split(aggregated_train, aggregated_train.Target, test_size = 0.2, random_state = 0, stratify = aggregated_train.Target)

# Check shapes of split data
X_train.shape, X_val.shape

In [ ]:
# Check target disribution of split data
X_train.Target.value_counts(normalize = True), X_val.Target.value_counts(normalize = True)

In [ ]:
X_train.head()

In [ ]:
# Instantiate and train model
model = LGBMClassifier(random_state = 0)

# Prepare data
X_train_data = X_train.drop(['ID', 'Target'], axis = 1)
X_val_data = X_val.drop(['ID', 'Target'], axis = 1)

# Train model
model.fit(X_train_data, y_train)

# Evaluate model
y_preds = model.predict(X_val_data)

# Measure model performance
val_f1_score = f1_score(y_val, y_preds, average = 'macro')
val_f1_score

In [ ]:
# Prepare test data
X_test = aggregated_test.drop(['ID'], axis = 1)

# Make predictions on test
test_preds = model.predict(X_test)

# Prepares sub file
sub = pd.DataFrame()
sub['ID'] = aggregated_test.ID
sub['Target'] = test_preds
sub.head()

In [ ]:
# Check sub target distribution
sub.Target.value_counts(normalize = True)

In [ ]:
# Create submission to be uploaded to Zindi for scoring
sub.to_csv('multiclass_benchmark.csv', index = False)